# 산업·사업모델 Agent · Neon 실데이터
분기 재무(`core.financial_quarter`)·공시(`core.disclosure`)·문서(`rag.document`)를 근거로 산업·사업모델 Agent를 실행합니다.

```
Neon(core.*, rag.*) ──조회──▶ 원자료 묶음 ──자동 변환──▶ report/evidence 입력 ──품질 검사──▶ Agent 실행
                                  (neon_source)        (neon_report)               (assess_report + neon_quality)
```

- 실행 위치: `OpenResearch_QualityAgents/notebooks` (프로젝트 루트의 `.env`에 `DATABASE_URL`).
- DB에는 쓰지 않습니다. 접속은 읽기 전용 세션이고, 접속 주소·비밀번호는 출력하지 않습니다.
- Neon 풀러 주소(`-pooler`)여도 코드가 직접 연결 주소로 바꿔 접속합니다.
- 품질 판정이 **보류**면 Agent 계산·LLM이 실행되지 않고 사유만 반환됩니다.

In [ ]:
# 최초 한 번만 (주석 해제)
# %pip install -r ../requirements-neon-agents.txt

In [ ]:
from pathlib import Path
import sys, os, json
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agent/neon_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("OpenResearch_QualityAgents 폴더(또는 그 안의 notebooks)에서 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)   # .env를 고친 뒤 커널 재시작 없이 다시 읽기
pd.set_option("display.max_colwidth", 120)

from agent.neon_source import connect, schema
from agent.neon_pipeline import prepare
from agent.neon_report import report_summary, today_kst
from agent.data_inventory import Inventory
from agent.quality_gate import gate_table, inventory_gate_table, NAMES
from agent.basic_agents import AgentSettings, DemoLLM
print("DATABASE_URL:", "있음" if os.getenv("DATABASE_URL") else "없음 → .env 확인",
      "| 풀러 주소:", "-pooler" in os.getenv("DATABASE_URL", ""), "(있어도 코드가 직접 연결 주소로 바꿔 접속)")

## 0. 설정

In [ ]:
TICKERS = ["005930", "000660"]   # 종목코드 문자열, 최대 20개
MARKET = "KR"
START = "2024-10-27"              # 조회 시작일 (시세·공시·문서 기준)
END = today_kst()                 # 조회 종료일 = 보고서 기준일. 과거 시점 분석이면 날짜로 고정
REF = today_kst()                 # 품질 검사 기준일 (최신성 판정)
WINDOW = 20                       # 이동평균 등 최소 관측 수. 품질 검사와 Agent가 같은 값을 써야 함
EXPECTED_ROLE = None              # 예: "agent_reader". None이면 역할 확인 생략(neondb_owner도 읽기 전용 세션으로 접속)
PRICE_ADJUSTMENT = None           # 산업 Agent는 시세를 쓰지 않음
PRICE_SOURCE_NOTE = None
DOMAIN, AGENT_NAME, SUFFIX = "industry_business_model", "산업·사업모델", "industry"
TABLES = ["core.financial_quarter", "core.financial_fact", "core.disclosure", "rag.document"]

## 1. DB 연결 확인
이 Agent가 쓰는 테이블의 실제 열 이름을 보여 줍니다. 열 이름이 다르면 이 표를 붙여 주세요.

In [ ]:
with connect(ROOT / ".env", expected_role=EXPECTED_ROLE) as conn:
    role = conn.execute("SELECT current_user AS r").fetchone()["r"]
    sch = schema(conn)
print("접속 성공 · 역할:", role, "· 읽기 전용 세션, 쿼리 15초 제한")
display(pd.DataFrame([{"테이블": t, "열": ", ".join(c)} for t, c in sch.items() if t in TABLES]))

## 2. DB 보유 현황 (집계 점검)
기존 품질 노트북과 같은 집계 검사입니다. 원자료 검사는 다음 단계에서 합니다.

In [ ]:
with connect(ROOT / ".env", expected_role=EXPECTED_ROLE) as conn:
    inv = Inventory(conn)
    companies = inv.companies(TICKERS, MARKET)
    coverage = inv.coverage(companies, START, END)
print("미조회 종목:", sorted(set(TICKERS) - {c["ticker"] for c in companies}) or "없음")
cov = pd.DataFrame(coverage)
display(cov[cov["테이블"].isin(TABLES)][["종목", "기업", "테이블", "상태", "건수", "최초", "최종", "필드 결측"]])
ready = pd.DataFrame(inventory_gate_table(coverage, start=START, end=END, reference_date=REF, window=WINDOW))
display(ready[ready["Agent"] == AGENT_NAME])

## 3. 원자료 조회 → 입력 형식 자동 변환 → 품질 검사
`prepare()`가 종목마다 Neon에서 읽고, 두 Agent 공통 입력(report/evidence)으로 바꾸고, 입력 계약 검사(assess_report)와 원자료 검사(neon_quality)를 합쳐 판정합니다.

In [ ]:
prepared = {}
with connect(ROOT / ".env", expected_role=EXPECTED_ROLE) as conn:
    for t in TICKERS:
        try:
            prepared[t] = prepare(conn, t, start=START, end=END, window=WINDOW, reference_date=REF,
                                  price_adjustment=PRICE_ADJUSTMENT, price_source_note=PRICE_SOURCE_NOTE)
        except LookupError as e:
            print("건너뜀:", e)

print("변환된 입력(report) 요약")
display(pd.DataFrame([report_summary(p.report) for p in prepared.values()]))
print("품질 판정 (실행 가능 / 일부 가능 / 보류)")
display(pd.DataFrame([row for p in prepared.values() for row in gate_table(p.gates) if row["Agent"] == AGENT_NAME]))

In [ ]:
# 규칙 상세: raw: 로 시작하면 DB 원자료 검사, 나머지는 입력 계약(assess_report) 검사
for t, p in prepared.items():
    checks = pd.DataFrame(p.gates[DOMAIN]["checks"])
    print(t, p.bundle["company"]["name"])
    display(checks[["규칙", "판정", "이유", "관측값"]])

### 변환된 입력 미리보기
분기 재무 표(E01)와 근거 목록입니다. 증감률·영업이익률은 코드가 계산하고, 사업모델 해석 주장(C02)은 원문 검수 전이므로 `withheld`로 들어갑니다.
지표 이름을 인식하지 못하면 `agent/neon_report.py`의 `METRIC_ALIASES`에 DB의 metric 이름을 추가하세요.

In [ ]:
from agent import neon_industry
for t, p in prepared.items():
    print(t, p.bundle["company"]["name"], "· 문서 본문 출처:", p.bundle.get("documentTextSource"))
    display(neon_industry.financial_frame(p))
    display(pd.DataFrame([c for c in p.report["calculations"] if c["calculationId"].startswith("CAL_FIN")])
            [["calculationId", "metric", "formula", "inputs", "result", "unit"]])

## 4. 산업·사업모델 Agent 실행

In [ ]:
# LLM_MODE: "none" = Python 분석만 / "demo" = 오프라인 고정 문구 / "solar" = Upstage Solar 실제 호출(비용 발생)
LLM_MODE = "demo"
if LLM_MODE == "solar":
    from agent.basic_agents import SolarLLM
    llm = SolarLLM()
elif LLM_MODE == "demo":
    llm = DemoLLM()
else:
    llm = None
print("LLM:", LLM_MODE)

In [ ]:
results = {}
for t, p in prepared.items():
    r = neon_industry.run(p, llm=llm, settings=AgentSettings(window=WINDOW, literacy="L2", style="U"))
    results[t] = r
    print("=" * 80, f"\n{t} {r['subject']['name']} · 상태: {r['status']} · 품질: {r['qualityGate']['label']}")
    if r["claims"]:
        display(pd.DataFrame(r["claims"])[["claimId", "kind", "assessment", "text"]])
    if r["calculations"]:
        display(pd.DataFrame(r["calculations"])[["calculationId", "formula", "result", "unit"]])
    display(neon_industry.source_frame(r).head(10))
    if "competitionTable" in r:
        display(pd.DataFrame(r["competitionTable"]))
    print("한계:", *r["limitations"], sep="\n- ")
    print("다음 요청:", *r["followUpRequests"], sep="\n- ")
    if r.get("explanation"):
        print("설명(" + r["explanationStatus"] + "):", r["explanation"])

## 5. 결과 저장 (선택)

In [ ]:
SAVE_RESULTS = False   # True면 private_outputs/neon_agents/ 에 입력 report와 결과 JSON 저장 (DB에는 쓰지 않음)
if SAVE_RESULTS:
    out = ROOT / "private_outputs/neon_agents"
    out.mkdir(parents=True, exist_ok=True)
    for t, r in results.items():
        (out / f"{t}_{SUFFIX}_input.json").write_text(
            json.dumps(prepared[t].report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
        (out / f"{t}_{SUFFIX}_result.json").write_text(
            json.dumps(r, ensure_ascii=False, indent=2, allow_nan=False, default=str), encoding="utf-8")
    print("저장:", out)